In [ ]:
import os
import jax
import corner
import matplotlib.pyplot as plt
from scipy.special import erf
import chime_frb_constants
import matplotlib.pyplot as plt
import corner
import pymc as pm
import numpy as np
import pytensor
import pytensor.tensor as pt
from scipy.special import erf
import arviz as az
import pymc.sampling.jax as pm_jax
import matplotlib.ticker as ticker
import time
import gc
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import jax

In [ ]:
NU_REF = chime_frb_constants.FREQ_BOTTOM_MHZ

In [ ]:
def get_times_freqs(data):
    freqs = np.linspace(chime_frb_constants.FREQ_BOTTOM_MHZ, chime_frb_constants.FREQ_TOP_MHZ, data.shape[0])
    times = np.arange(data.shape[1])*chime_frb_constants.SAMPLING_TIME_MS
    return times, freqs

In [ ]:
def arrival_time_estimate(data, times):
    integrated_profile = np.mean(data, axis=0)
    return times[np.argmax(integrated_profile)]

In [ ]:
# --- 1. Spectral Model
def spectral_energy_distribution_rpl(freqs, ref_freq, beta, gamma):
    f_vec = pt.as_tensor_variable(freqs.astype("float32"))[:, None]
    nu_ratio = f_vec / ref_freq
    # Returns F_k: weight for each channel
    return (nu_ratio)**(gamma + beta * pt.log(nu_ratio))

# --- 2. Temporal Model: No scattering / Gaussian
def gaussian_profile_pt(times, t0, sigma):
    t_vec = pt.as_tensor_variable(times.astype("float32"))[None, :]
    dt = t_vec - t0
    # Returns a symmetric shape centered at t0
    return pt.exp(-0.5 * (dt / sigma)**2)

# --- 3. Temporal Model: Scattering / McKinnon
def mckinnon_profile_pt(times, freqs, tau_r, t0, sigma, ref_freq, scattering_index=-4.0):
    t_vec = pt.as_tensor_variable(times.astype("float32"))[None, :]
    f_vec = pt.as_tensor_variable(freqs.astype("float32"))[:, None]
    nu_ratio = f_vec / ref_freq
    
    # Calculate tau_k (how much each channel smears)
    tau_k = tau_r * (nu_ratio)**(scattering_index)
    tau_k = pt.maximum(tau_k, 1e-9) # Safety buffer for 0
    
    dt = t_vec - t0
    inv_tau = 1.0 / tau_k
    
    # Standard McKinnon Math
    term1 = (sigma * pt.sqrt(pt.pi / 2)) * inv_tau
    exponent = (sigma**2 / (2 * tau_k**2)) - (dt * inv_tau)
    term2 = pt.exp(pt.clip(exponent, -700, 700))
    erf_arg = (dt - (sigma**2 * inv_tau)) / (sigma * pt.sqrt(2))
    term3 = 1 + pt.erf(erf_arg)
    
    return term1 * term2 * term3

In [ ]:
def model_fit(data, 
             scattering=False,
              model_scattering_index=False,
              scattering_index=-4,
              draws=4000,
              tune=2000,
              chains=4,
              nuts_sampler="blackjax",
              
             ):
    times, freqs = get_times_freqs(data)
    t0_peak_guess = arrival_time_estimate(data, times)
    with pm.Model() as frb_model_no_scattering:
        noise_std = np.std(data[:, :20]).astype("float32")
        sqrt2 = np.sqrt(2).astype("float32")
        
        # --- PRIORS ---
        alpha = pm.Normal("alpha", mu=0, sigma=2)
        amp_base = pm.Deterministic("amp_base", 10**alpha)
        t0 = pm.Uniform("t0", lower=t0_peak_guess - 2, upper=t0_peak_guess + 2)
        sigma = pm.TruncatedNormal("sigma", mu=1.0, sigma=0.5, lower=0.1)
        gamma = pm.Normal("gamma", mu=0, sigma=10)
        beta = pm.Normal("beta", mu=0, sigma=50)
        
        
        F_k = spectral_energy_distribution_rpl(freqs, NU_REF, beta, gamma)
        if scattering:
            tau_r = pm.Uniform("tau_r", lower=0.1, upper=100)
            if model_scattering_index:
                scattering_index = pm.Uniform("theta", lower=-5, upper=-3)
            else:
                scattering_index = scattering_index
            profile = mckinnon_profile_pt(times, freqs, tau_r, t0, sigma, NU_REF, scattering_index)
            
        else:
            profile = gaussian_profile_pt(times, t0, sigma)
    
        M_kn = pm.Deterministic("M_kn", (10**alpha) * F_k * profile)
        
        likelihood = pm.Normal("obs", mu=M_kn, sigma=noise_std, observed=data)

        trace = pm.sample(
            draws=draws, 
            tune=tune,
            chains=chains,
            progressbar=False,   # False required for vectorized methods
            nuts_sampler=nuts_sampler,
            nuts_sampler_kwargs={
                "chain_method": "vectorized" # vectorized or parallel
            }
        )

        return trace, likelihood

In [ ]:
data = np.load("scat_time.npy")

In [ ]:
def get_estimate(trace, parameter, credible_interval=95):
    try:
        posterior = trace.posterior[parameter]
        median = np.median(posterior)
        alpha = 1 - (credible_interval / 100)
        half_alpha = alpha / 2
        lower_bound = np.quantile(posterior, half_alpha)
        upper_bound =  np.quantile(posterior, (1- half_alpha))
    
        return median.item(), lower_bound.item(), upper_bound.item()
    except Exception:
        raise ValueError(f"The parameter: '{parameter}' wasn't either modeled or misspelled. The valid parameters are: ['alpha', 't0', 'sigma', 'gamma', 'beta', 'tau_r', 'theta']" )

In [ ]:
files = Path("npy_files/")

In [ ]:
list_files = list(files.glob("*.npy"))

In [ ]:
filename = []
median = []
lower = []
upper = []

for file in tqdm(list_files):
    name = file.name
    data = np.load(file)
    trace, likelihood = model_fit(data, scattering=True)
    med, low, high = get_estimate(trace, "tau_r", credible_interval=95)
    filename.append(name)
    median.append(med)
    lower.append(med-low)
    upper.append(high-med)
    del data, trace, likelihood
    gc.collect()
    jax.clear_caches()

In [ ]:
mcmc_results = {"filenames":filename, "median": median, "lower": lower, "uppder": upper}

In [ ]:
df = pd.DataFrame(mcmc_results)

In [ ]:
df

In [ ]:
df.to_csv("mcmc_results.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(np.load(list_files[43]), aspect='auto', origin='lower')

In [ ]:
plt.plot(np.load(list_files[30]).sum(axis=0))

In [ ]:
plt.scatter(np.arange(2, 51), median)